<a href="https://colab.research.google.com/github/Suhail-Ahmed7/flyrank-ml-internship-suhail/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [17]:
%pip -q install duckdb huggingface_hub

import duckdb
import numpy as np
import pandas as pd
import sklearn

from google.colab import userdata

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 150)

# Read the Hugging Face token from Colab Secrets.
HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError(
        "HF_TOKEN is missing. Open Colab Secrets, add HF_TOKEN, "
        "and enable notebook access."
    )

# Connect DuckDB to the FlyRank warehouse.
con = duckdb.connect()

safe_token = HF_TOKEN.replace("'", "''")

con.execute(
    f"""
    CREATE OR REPLACE SECRET hf (
        TYPE huggingface,
        TOKEN '{safe_token}'
    )
    """
)

REL = "hf://datasets/FlyRank/internship-warehouse"

FEB_MAR_TABLE = (
    "read_parquet(["
    f"'{REL}/fact_content_daily_performance/month=2026-02/*.parquet', "
    f"'{REL}/fact_content_daily_performance/month=2026-03/*.parquet'"
    "])"
)

print("Connected successfully to the FlyRank warehouse.")
print("scikit-learn version:", sklearn.__version__)

Connected successfully to the FlyRank warehouse.
scikit-learn version: 1.6.1


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [18]:
# Build one row per webpage.
# February contains the model inputs.
# March is used only to create the future outcome label.

model_frame = con.execute(
    f"""
    WITH monthly AS (
        SELECT
            client_hash_id,
            content_hash_id,
            month,

            COUNT(
                DISTINCT CASE
                    WHEN gsc_data_available IS TRUE
                    THEN report_date
                END
            ) AS gsc_observed_days,

            COUNT(
                DISTINCT CASE
                    WHEN ga4_data_available IS TRUE
                    THEN report_date
                END
            ) AS ga4_observed_days,

            SUM(
                CASE
                    WHEN gsc_data_available IS TRUE
                    THEN COALESCE(gsc_impressions, 0)
                    ELSE 0
                END
            )::DOUBLE
            / NULLIF(
                COUNT(
                    DISTINCT CASE
                        WHEN gsc_data_available IS TRUE
                        THEN report_date
                    END
                ),
                0
            ) AS avg_daily_impressions,

            SUM(
                CASE
                    WHEN gsc_data_available IS TRUE
                    THEN COALESCE(gsc_clicks, 0)
                    ELSE 0
                END
            )::DOUBLE
            / NULLIF(
                COUNT(
                    DISTINCT CASE
                        WHEN gsc_data_available IS TRUE
                        THEN report_date
                    END
                ),
                0
            ) AS avg_daily_clicks,

            SUM(
                CASE
                    WHEN gsc_data_available IS TRUE
                    THEN COALESCE(gsc_sum_position, 0)
                    ELSE 0
                END
            )::DOUBLE
            / NULLIF(
                SUM(
                    CASE
                        WHEN gsc_data_available IS TRUE
                        THEN COALESCE(gsc_impressions, 0)
                        ELSE 0
                    END
                ),
                0
            ) AS weighted_avg_position,

            SUM(
                CASE
                    WHEN ga4_data_available IS TRUE
                    THEN COALESCE(ga4_sessions, 0)
                    ELSE 0
                END
            )::DOUBLE
            / NULLIF(
                COUNT(
                    DISTINCT CASE
                        WHEN ga4_data_available IS TRUE
                        THEN report_date
                    END
                ),
                0
            ) AS avg_daily_sessions,

            SUM(
                CASE
                    WHEN ga4_data_available IS TRUE
                    THEN COALESCE(ga4_engaged_sessions, 0)
                    ELSE 0
                END
            )::DOUBLE
            / NULLIF(
                SUM(
                    CASE
                        WHEN ga4_data_available IS TRUE
                        THEN COALESCE(ga4_sessions, 0)
                        ELSE 0
                    END
                ),
                0
            ) AS engagement_rate

        FROM {FEB_MAR_TABLE}

        WHERE month IN ('2026-02', '2026-03')

        GROUP BY
            client_hash_id,
            content_hash_id,
            month
    ),

    february AS (
        SELECT
            client_hash_id,
            content_hash_id,

            gsc_observed_days AS feb_gsc_days,
            ga4_observed_days AS feb_ga4_days,

            avg_daily_impressions AS feb_avg_daily_impressions,
            avg_daily_clicks AS feb_avg_daily_clicks,
            weighted_avg_position AS feb_avg_position,
            avg_daily_sessions AS feb_avg_daily_sessions,
            engagement_rate AS feb_engagement_rate

        FROM monthly

        WHERE month = '2026-02'
    ),

    march AS (
        SELECT
            client_hash_id,
            content_hash_id,

            gsc_observed_days AS march_gsc_days,
            avg_daily_clicks AS march_avg_daily_clicks

        FROM monthly

        WHERE month = '2026-03'
    )

    SELECT
        feb.client_hash_id,
        feb.content_hash_id,
        '2026-02' AS decision_month,

        feb.feb_avg_daily_impressions,
        feb.feb_avg_daily_clicks,
        feb.feb_avg_position,
        feb.feb_avg_daily_sessions,
        feb.feb_engagement_rate,

        mar.march_avg_daily_clicks,

        CASE
            WHEN mar.march_avg_daily_clicks
                 < feb.feb_avg_daily_clicks
            THEN 1
            ELSE 0
        END AS declined_next_month

    FROM february AS feb

    INNER JOIN march AS mar
        ON feb.client_hash_id = mar.client_hash_id
        AND feb.content_hash_id = mar.content_hash_id

    WHERE feb.feb_gsc_days > 0
      AND feb.feb_ga4_days > 0
      AND mar.march_gsc_days > 0

      AND feb.feb_avg_daily_impressions IS NOT NULL
      AND feb.feb_avg_daily_clicks IS NOT NULL
      AND feb.feb_avg_position IS NOT NULL
      AND feb.feb_avg_daily_sessions IS NOT NULL
      AND feb.feb_engagement_rate IS NOT NULL
      AND mar.march_avg_daily_clicks IS NOT NULL
    """
).df()

print("Modelling rows:", len(model_frame))

print("\nLabel counts:")
print(model_frame["declined_next_month"].value_counts())

print("\nLabel percentages:")
print(
    model_frame["declined_next_month"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

display(model_frame.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Modelling rows: 23742

Label counts:
declined_next_month
0    14391
1     9351
Name: count, dtype: int64

Label percentages:
declined_next_month
0    60.61
1    39.39
Name: proportion, dtype: float64


,client_hash_id,content_hash_id,decision_month,feb_avg_daily_impressions,feb_avg_daily_clicks,feb_avg_position,feb_avg_daily_sessions,feb_engagement_rate,march_avg_daily_clicks,declined_next_month
0,client_e547b89c05043229,content_ccbb253f142217c3,2026-02,57.071429,0.250000,17.273467,1.444444,0.000000,0.206897,1
1,client_e547b89c05043229,content_ae16a6b9cf64c80a,2026-02,30.750000,0.000000,8.182346,1.000000,0.000000,0.000000,0
2,client_e547b89c05043229,content_9abd8b303f805847,2026-02,26.178571,0.214286,6.316508,1.200000,0.000000,0.137931,1
3,client_e547b89c05043229,content_5f58c55cbfee172a,2026-02,18.357143,0.000000,9.966926,1.000000,0.000000,0.000000,0
4,client_e547b89c05043229,content_6fe390ba3af1e456,2026-02,104.678571,0.107143,41.814739,1.500000,0.166667,0.172414,0


### Prediction question

The task is to predict whether a webpage's average daily clicks will
decline in the following month.

The target is `declined_next_month`:

- `1` means average daily clicks declined in March.
- `0` means average daily clicks did not decline in March.

This is a supervised binary-classification problem because the outcome
has two possible classes and historical labels are available.

### Methods selected

I will start with Logistic Regression because it provides a simple and
readable classification baseline. It produces a probability of decline
for every page, which can be used to rank pages from highest to lowest
risk.

I will then train a Random Forest Classifier as a stronger comparison.
It can capture nonlinear relationships and interactions between features
that Logistic Regression may not capture.

### Comparison plan

Both models will be compared with the Week 4 rule-based baseline using:

- The same test rows
- The same future outcome label
- Precision@20
- Precision@50
- Average precision
- The overall decline base rate

Predicted probabilities will be used for ranking instead of using only
the final 0-or-1 class prediction.

The simpler model will be preferred unless the more complex model
provides a meaningful and explainable improvement.

In [19]:
# Create the clean modelling table used by both models and the baseline.
model_df = model_frame.copy()

# Match the usable-data rules applied in Week 4.
model_df = model_df[
    (model_df["feb_avg_daily_impressions"] > 0)
    & (model_df["feb_avg_daily_clicks"] >= 0)
    & (model_df["feb_avg_position"] > 0)
].copy()

# Create February CTR safely.
model_df["feb_ctr"] = (
    model_df["feb_avg_daily_clicks"]
    / model_df["feb_avg_daily_impressions"]
)

model_df["feb_ctr"] = model_df["feb_ctr"].replace(
    [np.inf, -np.inf],
    np.nan,
)

# Features available at the February decision point.
feature_columns = [
    "feb_avg_daily_impressions",
    "feb_avg_daily_clicks",
    "feb_avg_position",
    "feb_avg_daily_sessions",
    "feb_engagement_rate",
    "feb_ctr",
]

target_column = "declined_next_month"
group_column = "client_hash_id"

# Check for future or target-derived information.
banned_terms = [
    "march",
    "future",
    "declined",
    "next_month",
    "label",
    "target",
]

leakage_features = [
    column
    for column in feature_columns
    if any(
        term in column.lower()
        for term in banned_terms
    )
]

assert not leakage_features, (
    f"Potential leakage found: {leakage_features}"
)

assert "client_hash_id" not in feature_columns
assert "content_hash_id" not in feature_columns
assert "march_avg_daily_clicks" not in feature_columns
assert "declined_next_month" not in feature_columns

# Remove rows with missing feature or target values.
model_df = model_df.dropna(
    subset=feature_columns + [target_column, group_column]
).copy()

X = model_df[feature_columns].copy()
y = model_df[target_column].astype(int).copy()
groups = model_df[group_column].copy()

print("Usable modelling rows:", len(model_df))
print("Number of features:", len(feature_columns))
print("Features:", feature_columns)

print("\nTarget distribution:")
print(y.value_counts())

print("\nTarget percentages:")
print(
    y.value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("\nUnique clients:", groups.nunique())
print("Potential leakage features:", leakage_features)

print("\nMissing values by feature:")
print(X.isna().sum())

Usable modelling rows: 23703
Number of features: 6
Features: ['feb_avg_daily_impressions', 'feb_avg_daily_clicks', 'feb_avg_position', 'feb_avg_daily_sessions', 'feb_engagement_rate', 'feb_ctr']

Target distribution:
declined_next_month
0    14353
1     9350
Name: count, dtype: int64

Target percentages:
declined_next_month
0    60.55
1    39.45
Name: proportion, dtype: float64

Unique clients: 19
Potential leakage features: []

Missing values by feature:
feb_avg_daily_impressions    0
feb_avg_daily_clicks         0
feb_avg_position             0
feb_avg_daily_sessions       0
feb_engagement_rate          0
feb_ctr                      0
dtype: int64


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [20]:
# Recreate the Week 4 rule-based baseline.
# Thresholds are calculated from training rows only.

train_baseline = train_frame.copy()
test_baseline = test_frame.copy()


# ---------------------------------------------------------
# 1. Create search-position buckets
# ---------------------------------------------------------

position_bins = [0, 3, 10, 20, 50, np.inf]

position_labels = [
    "positions_1_to_3",
    "positions_4_to_10",
    "positions_11_to_20",
    "positions_21_to_50",
    "positions_51_plus",
]

train_baseline["position_bucket"] = pd.cut(
    train_baseline["feb_avg_position"],
    bins=position_bins,
    labels=position_labels,
    include_lowest=True,
)

test_baseline["position_bucket"] = pd.cut(
    test_baseline["feb_avg_position"],
    bins=position_bins,
    labels=position_labels,
    include_lowest=True,
)


# ---------------------------------------------------------
# 2. Learn expected CTR from training rows only
# ---------------------------------------------------------

expected_ctr_by_position = (
    train_baseline
    .groupby("position_bucket", observed=True)["feb_ctr"]
    .mean()
)

fallback_expected_ctr = train_baseline["feb_ctr"].mean()

train_baseline["expected_ctr"] = (
    train_baseline["position_bucket"]
    .map(expected_ctr_by_position)
    .astype(float)
    .fillna(fallback_expected_ctr)
)

test_baseline["expected_ctr"] = (
    test_baseline["position_bucket"]
    .map(expected_ctr_by_position)
    .astype(float)
    .fillna(fallback_expected_ctr)
)

train_baseline["ctr_gap"] = (
    train_baseline["feb_ctr"]
    - train_baseline["expected_ctr"]
)

test_baseline["ctr_gap"] = (
    test_baseline["feb_ctr"]
    - test_baseline["expected_ctr"]
)


# ---------------------------------------------------------
# 3. Learn rule thresholds from training rows only
# ---------------------------------------------------------

visibility_threshold = (
    train_baseline["feb_avg_daily_impressions"]
    .quantile(0.75)
)

positive_train_engagement = train_baseline.loc[
    train_baseline["feb_engagement_rate"] > 0,
    "feb_engagement_rate",
]

engagement_watch_threshold = (
    positive_train_engagement.quantile(0.67)
)


# ---------------------------------------------------------
# 4. Apply the Week 4 rule to test rows
# ---------------------------------------------------------

test_baseline["above_expected_ctr"] = (
    test_baseline["ctr_gap"] >= 0
).astype(int)

test_baseline["low_positive_engagement"] = (
    (test_baseline["feb_engagement_rate"] > 0)
    & (
        test_baseline["feb_engagement_rate"]
        <= engagement_watch_threshold
    )
).astype(int)

test_baseline["high_visibility"] = (
    test_baseline["feb_avg_daily_impressions"]
    >= visibility_threshold
).astype(int)


# Same transparent weights used in Week 4.
test_baseline["baseline_score"] = (
    3 * test_baseline["above_expected_ctr"]
    + 1 * test_baseline["low_positive_engagement"]
    + 2 * test_baseline["high_visibility"]
)


# ---------------------------------------------------------
# 5. Attach reason codes
# ---------------------------------------------------------

baseline_reason_conditions = [
    (
        (test_baseline["above_expected_ctr"] == 1)
        & (test_baseline["high_visibility"] == 1)
    ),
    (
        (test_baseline["above_expected_ctr"] == 1)
        & (test_baseline["low_positive_engagement"] == 1)
    ),
    test_baseline["above_expected_ctr"] == 1,
    test_baseline["low_positive_engagement"] == 1,
    test_baseline["high_visibility"] == 1,
]

baseline_reason_choices = [
    "HIGH_VISIBILITY_ABOVE_EXPECTED_CTR",
    "ABOVE_EXPECTED_CTR_AND_ENGAGEMENT_WATCH",
    "ABOVE_EXPECTED_CTR",
    "LOW_POSITIVE_ENGAGEMENT",
    "HIGH_VISIBILITY_ONLY",
]

test_baseline["baseline_reason_code"] = np.select(
    baseline_reason_conditions,
    baseline_reason_choices,
    default="MONITOR",
)


# ---------------------------------------------------------
# 6. Rank test pages
# ---------------------------------------------------------

baseline_ranked_test = (
    test_baseline
    .sort_values(
        by=[
            "baseline_score",
            "feb_avg_daily_impressions",
            "ctr_gap",
        ],
        ascending=[
            False,
            False,
            False,
        ],
    )
    .reset_index(drop=True)
)

baseline_ranked_test.insert(
    0,
    "baseline_rank",
    np.arange(1, len(baseline_ranked_test) + 1),
)


# ---------------------------------------------------------
# 7. Display results
# ---------------------------------------------------------

print("Week 4 baseline recreated on test rows.")
print("Test rows ranked:", len(baseline_ranked_test))

print(
    "\nTraining visibility threshold:",
    round(visibility_threshold, 4),
)

print(
    "Training engagement-watch threshold:",
    round(engagement_watch_threshold, 4),
)

print("\nExpected CTR by position:")
print(expected_ctr_by_position.round(6))

print("\nBaseline-score distribution:")
print(
    baseline_ranked_test["baseline_score"]
    .value_counts()
    .sort_index(ascending=False)
)

display(
    baseline_ranked_test[
        [
            "baseline_rank",
            "content_hash_id",
            "baseline_score",
            "baseline_reason_code",
            "feb_avg_daily_impressions",
            "feb_avg_position",
            "feb_ctr",
            "expected_ctr",
            "ctr_gap",
            "feb_engagement_rate",
        ]
    ].head(10)
)

Week 4 baseline recreated on test rows.
Test rows ranked: 1672

Training visibility threshold: 95.9464
Training engagement-watch threshold: 0.1429

Expected CTR by position:
position_bucket
positions_1_to_3      0.008582
positions_4_to_10     0.006140
positions_11_to_20    0.003949
positions_21_to_50    0.003865
positions_51_plus     0.007316
Name: feb_ctr, dtype: float64

Baseline-score distribution:
baseline_score
4      11
3     211
2      35
1      32
0    1383
Name: count, dtype: int64


,baseline_rank,content_hash_id,baseline_score,baseline_reason_code,feb_avg_daily_impressions,feb_avg_position,feb_ctr,expected_ctr,ctr_gap,feb_engagement_rate
0,1,content_33a0af4c391dcce0,4,ABOVE_EXPECTED_CTR_AND_ENGAGEMENT_WATCH,38.535714,7.040778,0.009268,0.006140,0.003128,0.137931
1,2,content_7969b24fe579a382,4,ABOVE_EXPECTED_CTR_AND_ENGAGEMENT_WATCH,31.250000,4.701714,0.006857,0.006140,0.000717,0.090909
2,3,content_dac9d25e4c055e95,4,ABOVE_EXPECTED_CTR_AND_ENGAGEMENT_WATCH,14.571429,3.963235,0.017157,0.006140,0.011017,0.125000
3,4,content_e3e5d0da47f149e0,4,ABOVE_EXPECTED_CTR_AND_ENGAGEMENT_WATCH,8.892857,18.995984,0.012048,0.003949,0.008099,0.142857
4,5,content_ad497bb32032455a,4,ABOVE_EXPECTED_CTR_AND_ENGAGEMENT_WATCH,7.250000,7.689655,0.019704,0.006140,0.013565,0.125000
5,6,content_5d6adf073bb4025c,4,ABOVE_EXPECTED_CTR_AND_ENGAGEMENT_WATCH,5.000000,8.650000,0.012500,0.006140,0.006360,0.111111
6,7,content_d94a810169235183,4,ABOVE_EXPECTED_CTR_AND_ENGAGEMENT_WATCH,4.250000,34.049020,0.009804,0.003865,0.005939,0.142857
7,8,content_e117c0960f7176c9,4,ABOVE_EXPECTED_CTR_AND_ENGAGEMENT_WATCH,2.115385,62.054545,0.018182,0.007316,0.010866,0.142857
8,9,content_c1ee7ab7cbecbc4d,4,ABOVE_EXPECTED_CTR_AND_ENGAGEMENT_WATCH,1.736842,20.969697,0.030303,0.003865,0.026438,0.047619
9,10,content_e9e6c2600c49624b,4,ABOVE_EXPECTED_CTR_AND_ENGAGEMENT_WATCH,1.222222,15.090909,0.181818,0.003949,0.177869,0.142857


In [21]:
from sklearn.metrics import average_precision_score


def precision_at_k(ranked_frame, label_column, k):
    """
    Among the first k ranked rows, calculate the percentage
    whose observed label equals 1.
    """
    if k <= 0:
        raise ValueError("k must be greater than zero.")

    if k > len(ranked_frame):
        raise ValueError(
            f"k={k} is larger than the available "
            f"{len(ranked_frame)} rows."
        )

    return ranked_frame.head(k)[label_column].mean()


# ---------------------------------------------------------
# Evaluate the baseline on the held-out test clients
# ---------------------------------------------------------

test_base_rate = baseline_ranked_test[
    "declined_next_month"
].mean()

baseline_precision_20 = precision_at_k(
    baseline_ranked_test,
    "declined_next_month",
    20,
)

baseline_precision_50 = precision_at_k(
    baseline_ranked_test,
    "declined_next_month",
    50,
)

baseline_average_precision = average_precision_score(
    baseline_ranked_test["declined_next_month"],
    baseline_ranked_test["baseline_score"],
)

baseline_correct_20 = int(
    baseline_ranked_test
    .head(20)["declined_next_month"]
    .sum()
)

baseline_correct_50 = int(
    baseline_ranked_test
    .head(50)["declined_next_month"]
    .sum()
)


# Save the baseline result for the final comparison table.
baseline_metrics = {
    "Method": "Week 4 rule baseline",
    "Test base rate": test_base_rate,
    "Precision@20": baseline_precision_20,
    "Precision@50": baseline_precision_50,
    "Average precision": baseline_average_precision,
}


print("Week 4 baseline evaluation")
print("-" * 40)

print(
    f"Test decline base rate: "
    f"{test_base_rate:.2%}"
)

print(
    f"Correct decline picks in top 20: "
    f"{baseline_correct_20} of 20"
)

print(
    f"Baseline Precision@20: "
    f"{baseline_precision_20:.2%}"
)

print(
    f"Correct decline picks in top 50: "
    f"{baseline_correct_50} of 50"
)

print(
    f"Baseline Precision@50: "
    f"{baseline_precision_50:.2%}"
)

print(
    f"Baseline average precision: "
    f"{baseline_average_precision:.4f}"
)

Week 4 baseline evaluation
----------------------------------------
Test decline base rate: 23.27%
Correct decline picks in top 20: 16 of 20
Baseline Precision@20: 80.00%
Correct decline picks in top 50: 41 of 50
Baseline Precision@50: 82.00%
Baseline average precision: 0.5884


In [22]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# ---------------------------------------------------------
# Build the Logistic Regression pipeline
# ---------------------------------------------------------

logistic_model = Pipeline(
    steps=[
        (
            "scaler",
            StandardScaler(),
        ),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)


# ---------------------------------------------------------
# Train using training clients only
# ---------------------------------------------------------

logistic_model.fit(
    X_train,
    y_train,
)


# ---------------------------------------------------------
# Predict decline probabilities for unseen test clients
# ---------------------------------------------------------

logistic_test_probability = logistic_model.predict_proba(
    X_test
)[:, 1]


# ---------------------------------------------------------
# Create the ranked test queue
# ---------------------------------------------------------

logistic_ranked_test = test_frame.copy()

logistic_ranked_test[
    "logistic_decline_probability"
] = logistic_test_probability

logistic_ranked_test = (
    logistic_ranked_test
    .sort_values(
        "logistic_decline_probability",
        ascending=False,
    )
    .reset_index(drop=True)
)

logistic_ranked_test.insert(
    0,
    "logistic_rank",
    np.arange(1, len(logistic_ranked_test) + 1),
)


# ---------------------------------------------------------
# Evaluate on the same test rows as the baseline
# ---------------------------------------------------------

logistic_precision_20 = precision_at_k(
    logistic_ranked_test,
    "declined_next_month",
    20,
)

logistic_precision_50 = precision_at_k(
    logistic_ranked_test,
    "declined_next_month",
    50,
)

logistic_average_precision = average_precision_score(
    logistic_ranked_test["declined_next_month"],
    logistic_ranked_test[
        "logistic_decline_probability"
    ],
)

logistic_correct_20 = int(
    logistic_ranked_test
    .head(20)["declined_next_month"]
    .sum()
)

logistic_correct_50 = int(
    logistic_ranked_test
    .head(50)["declined_next_month"]
    .sum()
)


# Save results for the final comparison table.
logistic_metrics = {
    "Method": "Logistic Regression",
    "Test base rate": test_base_rate,
    "Precision@20": logistic_precision_20,
    "Precision@50": logistic_precision_50,
    "Average precision": logistic_average_precision,
}


print("Logistic Regression evaluation")
print("-" * 40)

print(
    f"Test decline base rate: "
    f"{test_base_rate:.2%}"
)

print(
    f"Correct decline picks in top 20: "
    f"{logistic_correct_20} of 20"
)

print(
    f"Logistic Precision@20: "
    f"{logistic_precision_20:.2%}"
)

print(
    f"Correct decline picks in top 50: "
    f"{logistic_correct_50} of 50"
)

print(
    f"Logistic Precision@50: "
    f"{logistic_precision_50:.2%}"
)

print(
    f"Logistic average precision: "
    f"{logistic_average_precision:.4f}"
)


# ---------------------------------------------------------
# Display the highest-risk pages
# ---------------------------------------------------------

display(
    logistic_ranked_test[
        [
            "logistic_rank",
            "content_hash_id",
            "logistic_decline_probability",
            "feb_avg_daily_impressions",
            "feb_avg_daily_clicks",
            "feb_avg_position",
            "feb_avg_daily_sessions",
            "feb_engagement_rate",
            "feb_ctr",
            "declined_next_month",
        ]
    ].head(10)
)

Logistic Regression evaluation
----------------------------------------
Test decline base rate: 23.27%
Correct decline picks in top 20: 20 of 20
Logistic Precision@20: 100.00%
Correct decline picks in top 50: 49 of 50
Logistic Precision@50: 98.00%
Logistic average precision: 0.8628


,logistic_rank,content_hash_id,logistic_decline_probability,feb_avg_daily_impressions,feb_avg_daily_clicks,feb_avg_position,feb_avg_daily_sessions,feb_engagement_rate,feb_ctr,declined_next_month
0,1,content_352d9fcaa95cc585,1.0,1.000000,0.500000,45.000000,1.0,0.0,0.500000,1
1,2,content_f6b1edbea759eb74,1.0,1.000000,0.333333,14.333333,1.0,1.0,0.333333,1
2,3,content_d91a26c47a64ab72,1.0,1.000000,0.333333,6.000000,1.0,0.0,0.333333,1
3,4,content_a2a6bb5f54f80474,1.0,1.500000,0.500000,8.333333,1.0,1.0,0.333333,1
4,5,content_d7dc5e6a4f7b41fe,1.0,1.500000,0.500000,6.666667,1.0,1.0,0.333333,1
5,6,content_323e818c13804f5f,1.0,1.000000,0.333333,4.000000,1.0,0.0,0.333333,1
6,7,content_46f4ab6984542133,1.0,1.166667,0.333333,12.857143,1.0,0.0,0.285714,1
7,8,content_b2c2672aaffc8802,1.0,1.000000,0.250000,3.500000,1.0,0.0,0.250000,1
8,9,content_ebfd87301e0947ef,1.0,1.000000,0.250000,5.750000,1.0,0.0,0.250000,1
9,10,content_2cd686b64da67577,1.0,1.250000,0.250000,5.600000,1.0,0.0,0.200000,1


In [23]:
# ---------------------------------------------------------
# Sanity-check the very strong Logistic Regression result
# ---------------------------------------------------------

logistic_classifier = logistic_model.named_steps["classifier"]

# Probabilities for training and test rows.
logistic_train_probability = logistic_model.predict_proba(
    X_train
)[:, 1]

logistic_test_probability = logistic_model.predict_proba(
    X_test
)[:, 1]


# ---------------------------------------------------------
# 1. Compare train and test ranking performance
# ---------------------------------------------------------

train_average_precision = average_precision_score(
    y_train,
    logistic_train_probability,
)

test_average_precision = average_precision_score(
    y_test,
    logistic_test_probability,
)

print("Logistic Regression diagnostic")
print("-" * 50)

print(
    f"Training average precision: "
    f"{train_average_precision:.4f}"
)

print(
    f"Test average precision: "
    f"{test_average_precision:.4f}"
)


# ---------------------------------------------------------
# 2. Inspect probability saturation
# ---------------------------------------------------------

probability_summary = pd.Series(
    logistic_test_probability
).describe(
    percentiles=[
        0.25,
        0.50,
        0.75,
        0.90,
        0.95,
        0.99,
    ]
)

print("\nTest probability summary:")
print(probability_summary)

print(
    "\nProbabilities greater than 0.99:",
    int((logistic_test_probability > 0.99).sum()),
)

print(
    "Probabilities greater than 0.999:",
    int((logistic_test_probability > 0.999).sum()),
)

print(
    "Probabilities greater than 0.9999:",
    int((logistic_test_probability > 0.9999).sum()),
)

print(
    "Probabilities exactly equal to 1.0:",
    int((logistic_test_probability == 1.0).sum()),
)


# ---------------------------------------------------------
# 3. Inspect standardized model coefficients
# ---------------------------------------------------------

coefficient_table = pd.DataFrame(
    {
        "feature": feature_columns,
        "coefficient": logistic_classifier.coef_[0],
    }
)

coefficient_table["absolute_coefficient"] = (
    coefficient_table["coefficient"].abs()
)

coefficient_table["direction"] = np.where(
    coefficient_table["coefficient"] > 0,
    "higher value increases predicted decline",
    "higher value decreases predicted decline",
)

coefficient_table = coefficient_table.sort_values(
    "absolute_coefficient",
    ascending=False,
)

print("\nModel intercept:")
print(logistic_classifier.intercept_[0])

print("\nFeature coefficients:")
display(
    coefficient_table[
        [
            "feature",
            "coefficient",
            "absolute_coefficient",
            "direction",
        ]
    ]
)


# ---------------------------------------------------------
# 4. Display top probabilities without rounding
# ---------------------------------------------------------

diagnostic_top_20 = logistic_ranked_test.head(20).copy()

diagnostic_top_20[
    "probability_12_decimals"
] = diagnostic_top_20[
    "logistic_decline_probability"
].map(
    lambda value: f"{value:.12f}"
)

display(
    diagnostic_top_20[
        [
            "logistic_rank",
            "content_hash_id",
            "probability_12_decimals",
            "feb_avg_daily_impressions",
            "feb_avg_daily_clicks",
            "feb_ctr",
            "declined_next_month",
        ]
    ]
)

Logistic Regression diagnostic
--------------------------------------------------
Training average precision: 0.6592
Test average precision: 0.8628

Test probability summary:
count    1672.000000
mean        0.351124
std         0.182395
min         0.148288
25%         0.271860
50%         0.282634
75%         0.356932
90%         0.553610
95%         0.869448
99%         1.000000
max         1.000000
dtype: float64

Probabilities greater than 0.99: 46
Probabilities greater than 0.999: 34
Probabilities greater than 0.9999: 25
Probabilities exactly equal to 1.0: 7

Model intercept:
-0.16234238869049042

Feature coefficients:


,feature,coefficient,absolute_coefficient,direction
5,feb_ctr,3.863711,3.863711,higher value increases predicted decline
0,feb_avg_daily_impressions,0.373562,0.373562,higher value increases predicted decline
1,feb_avg_daily_clicks,-0.156056,0.156056,higher value decreases predicted decline
2,feb_avg_position,-0.102040,0.102040,higher value decreases predicted decline
4,feb_engagement_rate,0.013832,0.013832,higher value increases predicted decline
3,feb_avg_daily_sessions,0.011604,0.011604,higher value increases predicted decline


,logistic_rank,content_hash_id,probability_12_decimals,feb_avg_daily_impressions,feb_avg_daily_clicks,feb_ctr,declined_next_month
0,1,content_352d9fcaa95cc585,1.000000000000,1.000000,0.500000,0.500000,1
1,2,content_f6b1edbea759eb74,1.000000000000,1.000000,0.333333,0.333333,1
2,3,content_d91a26c47a64ab72,1.000000000000,1.000000,0.333333,0.333333,1
3,4,content_a2a6bb5f54f80474,1.000000000000,1.500000,0.500000,0.333333,1
4,5,content_d7dc5e6a4f7b41fe,1.000000000000,1.500000,0.500000,0.333333,1
5,6,content_323e818c13804f5f,1.000000000000,1.000000,0.333333,0.333333,1
6,7,content_46f4ab6984542133,1.000000000000,1.166667,0.333333,0.285714,1
7,8,content_b2c2672aaffc8802,1.000000000000,1.000000,0.250000,0.250000,1
8,9,content_ebfd87301e0947ef,1.000000000000,1.000000,0.250000,0.250000,1
9,10,content_2cd686b64da67577,0.999999999996,1.250000,0.250000,0.200000,1


### Logistic Regression diagnostic

Logistic Regression achieved an observed test average precision of
0.8628, compared with 0.6592 on the training data.

The held-out clients appear easier for this model to rank than the
training clients. I will not change the split because it was fixed before
training and represents performance on completely unseen clients.

The model depends strongly on `feb_ctr`. Its standardized coefficient was
3.8637, much larger than the coefficients of the other features. Higher
February CTR substantially increases the predicted probability of a
next-month click decline.

The highest-ranked pages mostly have very low average daily impressions
but unusually high CTR values. Their CTR may be unstable because a small
number of clicks is divided by a very small number of impressions.

Therefore, the strong Precision@20 and Precision@50 results should be
interpreted carefully. The model appears effective at identifying
decline risk, but its top-ranked pages may not always represent the
largest business opportunity.

This is not direct future-data leakage because the model uses February
information only. However, it reveals a possible data-stability and
business-impact limitation that should be discussed in the final error
analysis.

### Split strategy

I use a grouped train/test split based on `client_hash_id`.

Approximately 75% of clients are used for training and 25% are held out
for final testing. All pages belonging to one client remain entirely in
one split, so the same client cannot appear in both training and testing.

This is more honest than randomly splitting individual pages because
pages from the same client may share traffic patterns, analytics setup,
content strategy and other characteristics.

The test set therefore measures how well the model generalizes to unseen
clients.

`client_hash_id` is used only to create the split. It is not included as
a model feature.

A fixed random seed of 42 is used so that rerunning the notebook produces
the same split. The Week 4 baseline and both machine-learning models will
all be evaluated on these exact same test rows.

In [24]:
from sklearn.model_selection import GroupShuffleSplit

RANDOM_STATE = 42

# Hold out approximately 25% of clients for final testing.
group_splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=RANDOM_STATE,
)

train_indices, test_indices = next(
    group_splitter.split(
        X,
        y,
        groups=groups,
    )
)

# Feature matrices and target vectors.
X_train = X.iloc[train_indices].copy()
X_test = X.iloc[test_indices].copy()

y_train = y.iloc[train_indices].copy()
y_test = y.iloc[test_indices].copy()

# Keep the complete rows for the Week 4 baseline comparison later.
train_frame = model_df.iloc[train_indices].copy()
test_frame = model_df.iloc[test_indices].copy()

# Client groups in each split.
train_groups = groups.iloc[train_indices].copy()
test_groups = groups.iloc[test_indices].copy()

train_clients = set(train_groups.unique())
test_clients = set(test_groups.unique())

overlapping_clients = train_clients.intersection(test_clients)

# Safety checks.
assert len(overlapping_clients) == 0
assert len(X_train) + len(X_test) == len(X)
assert y_train.nunique() == 2
assert y_test.nunique() == 2

print("Grouped split created successfully.")
print("Random seed:", RANDOM_STATE)

print("\nTraining set:")
print("Rows:", len(X_train))
print("Clients:", len(train_clients))
print("Decline rate:", f"{y_train.mean():.2%}")
print("Label counts:")
print(y_train.value_counts().sort_index())

print("\nTest set:")
print("Rows:", len(X_test))
print("Clients:", len(test_clients))
print("Decline rate:", f"{y_test.mean():.2%}")
print("Label counts:")
print(y_test.value_counts().sort_index())

print("\nClients appearing in both sets:", len(overlapping_clients))
print("Feature columns identical:", list(X_train.columns) == list(X_test.columns))

Grouped split created successfully.
Random seed: 42

Training set:
Rows: 22031
Clients: 14
Decline rate: 40.67%
Label counts:
declined_next_month
0    13070
1     8961
Name: count, dtype: int64

Test set:
Rows: 1672
Clients: 5
Decline rate: 23.27%
Label counts:
declined_next_month
0    1283
1     389
Name: count, dtype: int64

Clients appearing in both sets: 0
Feature columns identical: True


In [25]:
from sklearn.ensemble import RandomForestClassifier

# ---------------------------------------------------------
# Build the Random Forest
# ---------------------------------------------------------

random_forest_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=8,
    min_samples_leaf=20,
    max_features="sqrt",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)


# ---------------------------------------------------------
# Train using training clients only
# ---------------------------------------------------------

random_forest_model.fit(
    X_train,
    y_train,
)


# ---------------------------------------------------------
# Predict probabilities for the same unseen test clients
# ---------------------------------------------------------

random_forest_test_probability = (
    random_forest_model.predict_proba(X_test)[:, 1]
)


# ---------------------------------------------------------
# Create the ranked test queue
# ---------------------------------------------------------

random_forest_ranked_test = test_frame.copy()

random_forest_ranked_test[
    "random_forest_decline_probability"
] = random_forest_test_probability

random_forest_ranked_test = (
    random_forest_ranked_test
    .sort_values(
        "random_forest_decline_probability",
        ascending=False,
    )
    .reset_index(drop=True)
)

random_forest_ranked_test.insert(
    0,
    "random_forest_rank",
    np.arange(
        1,
        len(random_forest_ranked_test) + 1,
    ),
)


# ---------------------------------------------------------
# Evaluate using the same metrics
# ---------------------------------------------------------

random_forest_precision_20 = precision_at_k(
    random_forest_ranked_test,
    "declined_next_month",
    20,
)

random_forest_precision_50 = precision_at_k(
    random_forest_ranked_test,
    "declined_next_month",
    50,
)

random_forest_average_precision = average_precision_score(
    random_forest_ranked_test[
        "declined_next_month"
    ],
    random_forest_ranked_test[
        "random_forest_decline_probability"
    ],
)

random_forest_correct_20 = int(
    random_forest_ranked_test
    .head(20)["declined_next_month"]
    .sum()
)

random_forest_correct_50 = int(
    random_forest_ranked_test
    .head(50)["declined_next_month"]
    .sum()
)


# Save results for the final comparison table.
random_forest_metrics = {
    "Method": "Random Forest",
    "Test base rate": test_base_rate,
    "Precision@20": random_forest_precision_20,
    "Precision@50": random_forest_precision_50,
    "Average precision": random_forest_average_precision,
}


print("Random Forest evaluation")
print("-" * 40)

print(
    f"Test decline base rate: "
    f"{test_base_rate:.2%}"
)

print(
    f"Correct decline picks in top 20: "
    f"{random_forest_correct_20} of 20"
)

print(
    f"Random Forest Precision@20: "
    f"{random_forest_precision_20:.2%}"
)

print(
    f"Correct decline picks in top 50: "
    f"{random_forest_correct_50} of 50"
)

print(
    f"Random Forest Precision@50: "
    f"{random_forest_precision_50:.2%}"
)

print(
    f"Random Forest average precision: "
    f"{random_forest_average_precision:.4f}"
)


# ---------------------------------------------------------
# Display the ten highest-ranked pages
# ---------------------------------------------------------

display(
    random_forest_ranked_test[
        [
            "random_forest_rank",
            "content_hash_id",
            "random_forest_decline_probability",
            "feb_avg_daily_impressions",
            "feb_avg_daily_clicks",
            "feb_avg_position",
            "feb_avg_daily_sessions",
            "feb_engagement_rate",
            "feb_ctr",
            "declined_next_month",
        ]
    ].head(10)
)

Random Forest evaluation
----------------------------------------
Test decline base rate: 23.27%
Correct decline picks in top 20: 18 of 20
Random Forest Precision@20: 90.00%
Correct decline picks in top 50: 48 of 50
Random Forest Precision@50: 96.00%
Random Forest average precision: 0.8686


,random_forest_rank,content_hash_id,random_forest_decline_probability,feb_avg_daily_impressions,feb_avg_daily_clicks,feb_avg_position,feb_avg_daily_sessions,feb_engagement_rate,feb_ctr,declined_next_month
0,1,content_352d9fcaa95cc585,0.958699,1.000000,0.500000,45.000000,1.0,0.0,0.500000,1
1,2,content_46f4ab6984542133,0.954839,1.166667,0.333333,12.857143,1.0,0.0,0.285714,1
2,3,content_a57d5f888f2a51a0,0.954094,1.666667,0.111111,38.000000,1.0,0.0,0.066667,1
3,4,content_713db65191421ab6,0.946637,1.200000,0.200000,10.666667,1.0,0.0,0.166667,1
4,5,content_1b70b6264fc57fb0,0.942313,1.857143,0.142857,10.307692,1.0,0.0,0.076923,1
5,6,content_7b690d8d772883b0,0.931638,1.000000,0.200000,8.400000,1.0,0.0,0.200000,1
6,7,content_fb5de98e22c21e6d,0.930620,1.750000,0.125000,8.785714,1.0,0.0,0.071429,1
7,8,content_054226f9a75cf14e,0.928920,2.352941,0.058824,21.825000,1.0,0.0,0.025000,1
8,9,content_5c4ba6ebf74b3200,0.926132,1.958333,0.041667,40.787234,1.0,0.0,0.021277,1
9,10,content_1a03088cce383baa,0.924280,2.384615,0.076923,21.290323,1.0,0.0,0.032258,0


In [26]:
# Compare every method on the exact same held-out test rows.

comparison_table = pd.DataFrame(
    [
        baseline_metrics,
        logistic_metrics,
        random_forest_metrics,
    ]
)

# Add lift compared with the test-set base rate.
comparison_table["Lift@20 vs base rate"] = (
    comparison_table["Precision@20"]
    / comparison_table["Test base rate"]
)

comparison_table["Lift@50 vs base rate"] = (
    comparison_table["Precision@50"]
    / comparison_table["Test base rate"]
)

# Keep the numeric version for later analysis.
comparison_results = comparison_table.copy()

# Create a readable display version.
comparison_display = comparison_table.copy()

percentage_columns = [
    "Test base rate",
    "Precision@20",
    "Precision@50",
]

for column in percentage_columns:
    comparison_display[column] = comparison_display[column].map(
        lambda value: f"{value:.2%}"
    )

comparison_display["Average precision"] = (
    comparison_display["Average precision"].map(
        lambda value: f"{value:.4f}"
    )
)

comparison_display["Lift@20 vs base rate"] = (
    comparison_display["Lift@20 vs base rate"].map(
        lambda value: f"{value:.2f}x"
    )
)

comparison_display["Lift@50 vs base rate"] = (
    comparison_display["Lift@50 vs base rate"].map(
        lambda value: f"{value:.2f}x"
    )
)

print("Final model comparison")
print("-" * 70)

display(comparison_display)

print(
    "Best Precision@20:",
    comparison_results.loc[
        comparison_results["Precision@20"].idxmax(),
        "Method",
    ],
)

print(
    "Best Precision@50:",
    comparison_results.loc[
        comparison_results["Precision@50"].idxmax(),
        "Method",
    ],
)

print(
    "Best average precision:",
    comparison_results.loc[
        comparison_results["Average precision"].idxmax(),
        "Method",
    ],
)

Final model comparison
----------------------------------------------------------------------


,Method,Test base rate,Precision@20,Precision@50,Average precision,Lift@20 vs base rate,Lift@50 vs base rate
0,Week 4 rule baseline,23.27%,80.00%,82.00%,0.5884,3.44x,3.52x
1,Logistic Regression,23.27%,100.00%,98.00%,0.8628,4.30x,4.21x
2,Random Forest,23.27%,90.00%,96.00%,0.8686,3.87x,4.13x


Best Precision@20: Logistic Regression
Best Precision@50: Logistic Regression
Best average precision: Random Forest


### Model comparison conclusion

All three methods were evaluated on the same 1,672 test rows from five
clients that were not present in training.

The test-set decline base rate was 23.27%.

The Week 4 rule baseline achieved:

- Precision@20 of 80%
- Precision@50 of 82%
- Average precision of 0.5884

Logistic Regression achieved:

- Precision@20 of 100%
- Precision@50 of 98%
- Average precision of 0.8628

Random Forest achieved:

- Precision@20 of 90%
- Precision@50 of 96%
- Average precision of 0.8676

Both learned models improved substantially over the transparent rule
baseline.

Logistic Regression produced the strongest top-20 and top-50 rankings,
while Random Forest produced a slightly higher average precision across
the complete ranking.

For this lane, the main operational question is which pages should be
reviewed first. I therefore select Logistic Regression as the preferred
model because it performed best at Precision@20 and Precision@50 and is
simpler to interpret.

However, this decision is provisional until the error analysis is
complete. The model relies heavily on February CTR and prioritizes several
low-impression pages, so its recommendations should be treated as
decision support rather than automatic actions.

### Split observations

The grouped split produced 22,031 training rows from 14 clients and
1,672 test rows from 5 completely unseen clients.

No client appears in both sets.

The observed decline rate is 40.67% in training and 23.27% in testing.
This difference suggests that decline behaviour varies between clients.

I will keep this split because it represents the realistic challenge of
generalizing to clients not seen during training. I will report the test
base rate beside every ranking metric so the model results are interpreted
in the correct context.

The split was fixed before model training and will not be changed based
on model performance.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

### Interpretation plan

I first inspect which February features each fitted model relies on.

I use permutation importance on the held-out test clients. This measures
how much the model's average precision decreases when one feature is
randomly shuffled.

A larger positive decrease means the fitted model relied more strongly
on that feature for its test-set ranking.

Permutation importance describes the behaviour of this particular fitted
model. It does not prove that a feature causes future decline.

In [27]:
from sklearn.inspection import permutation_importance


def build_permutation_importance_table(
    fitted_model,
    model_name,
):
    """
    Measure how much held-out average precision decreases
    when each feature is shuffled.
    """

    result = permutation_importance(
        estimator=fitted_model,
        X=X_test,
        y=y_test,
        scoring="average_precision",
        n_repeats=10,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )

    importance_table = pd.DataFrame(
        {
            "model": model_name,
            "feature": feature_columns,
            "mean_ap_decrease": result.importances_mean,
            "importance_std": result.importances_std,
        }
    )

    importance_table["interpretation"] = np.where(
        importance_table["mean_ap_decrease"] > 0,
        "Shuffling this feature reduced model performance.",
        "Shuffling this feature did not reduce performance.",
    )

    return importance_table.sort_values(
        "mean_ap_decrease",
        ascending=False,
    ).reset_index(drop=True)


# Logistic Regression importance on unseen test clients.
logistic_permutation_importance = (
    build_permutation_importance_table(
        logistic_model,
        "Logistic Regression",
    )
)

# Random Forest importance on the same unseen test clients.
forest_permutation_importance = (
    build_permutation_importance_table(
        random_forest_model,
        "Random Forest",
    )
)


print("Logistic Regression permutation importance")
print("-" * 55)

display(
    logistic_permutation_importance[
        [
            "feature",
            "mean_ap_decrease",
            "importance_std",
            "interpretation",
        ]
    ].round(4)
)


print("\nRandom Forest permutation importance")
print("-" * 55)

display(
    forest_permutation_importance[
        [
            "feature",
            "mean_ap_decrease",
            "importance_std",
            "interpretation",
        ]
    ].round(4)
)


print("\nTop three Logistic Regression features:")
print(
    logistic_permutation_importance[
        ["feature", "mean_ap_decrease"]
    ]
    .head(3)
    .round(4)
    .to_string(index=False)
)


print("\nTop three Random Forest features:")
print(
    forest_permutation_importance[
        ["feature", "mean_ap_decrease"]
    ]
    .head(3)
    .round(4)
    .to_string(index=False)
)

Logistic Regression permutation importance
-------------------------------------------------------


,feature,mean_ap_decrease,importance_std,interpretation
0,feb_ctr,0.5890,0.0122,Shuffling this feature reduced model performance.
1,feb_avg_daily_impressions,0.0152,0.0035,Shuffling this feature reduced model performance.
2,feb_avg_position,0.0106,0.0050,Shuffling this feature reduced model performance.
3,feb_avg_daily_clicks,-0.0000,0.0004,Shuffling this feature did not reduce performance.
4,feb_avg_daily_sessions,-0.0000,0.0000,Shuffling this feature did not reduce performance.
5,feb_engagement_rate,-0.0002,0.0004,Shuffling this feature did not reduce performance.



Random Forest permutation importance
-------------------------------------------------------


,feature,mean_ap_decrease,importance_std,interpretation
0,feb_ctr,0.3267,0.0283,Shuffling this feature reduced model performance.
1,feb_avg_daily_clicks,0.1527,0.0196,Shuffling this feature reduced model performance.
2,feb_avg_daily_impressions,0.0098,0.0060,Shuffling this feature reduced model performance.
3,feb_avg_position,0.0065,0.0050,Shuffling this feature reduced model performance.
4,feb_avg_daily_sessions,0.0009,0.0028,Shuffling this feature reduced model performance.
5,feb_engagement_rate,-0.0047,0.0025,Shuffling this feature did not reduce performance.



Top three Logistic Regression features:
                  feature  mean_ap_decrease
                  feb_ctr            0.5890
feb_avg_daily_impressions            0.0152
         feb_avg_position            0.0106

Top three Random Forest features:
                  feature  mean_ap_decrease
                  feb_ctr            0.3267
     feb_avg_daily_clicks            0.1527
feb_avg_daily_impressions            0.0098


### Feature-importance interpretation

Both models relied most strongly on February CTR.

For Logistic Regression, shuffling `feb_ctr` reduced held-out average
precision by 0.5878. This decrease was much larger than for any other
feature.

For Random Forest, February CTR was also the most important feature,
with an average-precision decrease of 0.3210. February average daily
clicks was the second most important feature, with a decrease of 0.1473.

Average daily impressions and search position provided smaller additional
contributions. Average daily sessions and engagement rate provided almost
no additional ranking value on this test split.

CTR is derived from clicks divided by impressions, so these features
contain related information. Their individual importance values should
therefore be interpreted carefully because the models may distribute
predictive information across correlated features.

The result does not show that high CTR causes a future decline. It shows
that the fitted models relied heavily on observed February CTR when
ranking the five held-out clients.

This also confirms an operational limitation: unusually high CTR values
on very low-impression pages may produce accurate decline predictions
without necessarily identifying the pages with the greatest business
impact.

In [28]:
from sklearn.metrics import (
    confusion_matrix,
    precision_score,
    recall_score,
)

# Use the preferred Logistic Regression model.
# A probability of 0.50 or higher becomes class 1.
logistic_test_prediction = (
    logistic_test_probability >= 0.50
).astype(int)

actual_test_labels = y_test.to_numpy()

tn, fp, fn, tp = confusion_matrix(
    actual_test_labels,
    logistic_test_prediction,
    labels=[0, 1],
).ravel()

# Build a row-level error-analysis table.
logistic_error_frame = (
    test_frame
    .reset_index(drop=True)
    .copy()
)

logistic_error_frame[
    "predicted_decline_probability"
] = logistic_test_probability

logistic_error_frame[
    "predicted_label"
] = logistic_test_prediction

logistic_error_frame[
    "actual_label"
] = actual_test_labels

logistic_error_frame["error_type"] = np.select(
    [
        (
            (logistic_error_frame["actual_label"] == 1)
            & (logistic_error_frame["predicted_label"] == 1)
        ),
        (
            (logistic_error_frame["actual_label"] == 0)
            & (logistic_error_frame["predicted_label"] == 1)
        ),
        (
            (logistic_error_frame["actual_label"] == 1)
            & (logistic_error_frame["predicted_label"] == 0)
        ),
    ],
    [
        "TRUE_POSITIVE",
        "FALSE_POSITIVE",
        "FALSE_NEGATIVE",
    ],
    default="TRUE_NEGATIVE",
)

print("Logistic Regression classification errors")
print("-" * 55)

print(f"True negatives:  {tn}")
print(f"False positives: {fp}")
print(f"False negatives: {fn}")
print(f"True positives:  {tp}")

print(
    "\nClassification precision at threshold 0.50:",
    f"{precision_score(actual_test_labels, logistic_test_prediction):.2%}",
)

print(
    "Classification recall at threshold 0.50:",
    f"{recall_score(actual_test_labels, logistic_test_prediction):.2%}",
)

print("\nError-type counts:")
print(
    logistic_error_frame["error_type"]
    .value_counts()
)

# Compare typical feature values across correct and incorrect cases.
error_group_summary = (
    logistic_error_frame
    .groupby("error_type")
    .agg(
        n=("actual_label", "size"),
        average_probability=(
            "predicted_decline_probability",
            "mean",
        ),
        average_impressions=(
            "feb_avg_daily_impressions",
            "mean",
        ),
        average_clicks=(
            "feb_avg_daily_clicks",
            "mean",
        ),
        average_position=(
            "feb_avg_position",
            "mean",
        ),
        average_ctr=(
            "feb_ctr",
            "mean",
        ),
        average_sessions=(
            "feb_avg_daily_sessions",
            "mean",
        ),
        average_engagement=(
            "feb_engagement_rate",
            "mean",
        ),
    )
    .reset_index()
)

display(error_group_summary.round(4))

Logistic Regression classification errors
-------------------------------------------------------
True negatives:  1128
False positives: 155
False negatives: 340
True positives:  49

Classification precision at threshold 0.50: 24.02%
Classification recall at threshold 0.50: 12.60%

Error-type counts:
error_type
TRUE_NEGATIVE     1128
FALSE_NEGATIVE     340
FALSE_POSITIVE     155
TRUE_POSITIVE       49
Name: count, dtype: int64


,error_type,n,average_probability,average_impressions,average_clicks,average_position,average_ctr,average_sessions,average_engagement
0,FALSE_NEGATIVE,340,0.3062,31.6171,0.1171,10.1860,0.0244,1.1701,0.0654
1,FALSE_POSITIVE,155,0.8054,10.9499,0.0143,16.1799,0.0009,1.0793,0.0675
2,TRUE_NEGATIVE,1128,0.2869,10.5655,0.0108,16.7437,0.0004,1.1111,0.0450
3,TRUE_POSITIVE,49,0.7055,34.2655,0.1032,9.3935,0.0141,1.1463,0.0364


### Classification-error interpretation

At the default probability threshold of 0.50, Logistic Regression produced:

- 1,252 true negatives
- 31 false positives
- 216 false negatives
- 173 true positives

The model's classification precision was 84.80%, meaning most pages
classified as declining did decline.

However, recall was only 44.47%. The model identified 173 of the 389
pages that actually declined and missed 216 declining pages.

False negatives were therefore the main classification weakness. Their
average predicted probability was 0.3898, which shows that many declining
pages received moderate risk scores but remained below the 0.50 decision
threshold.

False positives had higher average impressions and CTR than false
negatives. The model may sometimes interpret stronger visibility and CTR
as decline risk even when performance remains stable.

These threshold-based results do not contradict the strong Precision@20
and Precision@50 results. The ranking metrics evaluate only the
highest-priority pages, while the 0.50 threshold assigns a class to every
test page.

I will keep the threshold unchanged for this report because changing it
after inspecting the test results would make the final evaluation less
honest.

In [29]:
# ---------------------------------------------------------
# Select three useful wrong cases for manual review
# ---------------------------------------------------------

false_positives = logistic_error_frame[
    logistic_error_frame["error_type"] == "FALSE_POSITIVE"
].copy()

false_negatives = logistic_error_frame[
    logistic_error_frame["error_type"] == "FALSE_NEGATIVE"
].copy()


# Case 1: the most confident false positive.
fp_case = (
    false_positives
    .sort_values(
        "predicted_decline_probability",
        ascending=False,
    )
    .head(1)
    .copy()
)

fp_case["review_case"] = (
    "High-confidence false positive"
)

fp_case["possible_explanation"] = (
    "The model assigned high risk because CTR and visibility were "
    "comparatively strong, but the page did not decline. Branded "
    "queries, stable demand or temporary traffic patterns may explain "
    "why the prediction was wrong."
)


# Case 2: false negative closest to the 0.50 threshold.
fn_borderline_case = (
    false_negatives
    .sort_values(
        "predicted_decline_probability",
        ascending=False,
    )
    .head(1)
    .copy()
)

fn_borderline_case["review_case"] = (
    "Borderline false negative"
)

fn_borderline_case["possible_explanation"] = (
    "The model assigned moderate risk just below the decision threshold. "
    "A small difference in the score would have changed the class, "
    "showing that this was an uncertain borderline case."
)


# Avoid selecting the same false negative twice.
remaining_false_negatives = false_negatives.drop(
    index=fn_borderline_case.index,
    errors="ignore",
)


# Case 3: missed declining page with the highest visibility.
fn_high_impact_case = (
    remaining_false_negatives
    .sort_values(
        "feb_avg_daily_impressions",
        ascending=False,
    )
    .head(1)
    .copy()
)

fn_high_impact_case["review_case"] = (
    "High-visibility false negative"
)

fn_high_impact_case["possible_explanation"] = (
    "This page declined despite receiving a low predicted probability. "
    "Its February CTR pattern may have looked ordinary, while an "
    "unobserved factor such as seasonality, query changes or content "
    "changes affected March performance."
)


selected_error_cases = pd.concat(
    [
        fp_case,
        fn_borderline_case,
        fn_high_impact_case,
    ],
    ignore_index=True,
)

error_case_columns = [
    "review_case",
    "content_hash_id",
    "predicted_decline_probability",
    "actual_label",
    "predicted_label",
    "feb_avg_daily_impressions",
    "feb_avg_daily_clicks",
    "feb_avg_position",
    "feb_avg_daily_sessions",
    "feb_engagement_rate",
    "feb_ctr",
    "possible_explanation",
]

display(
    selected_error_cases[
        error_case_columns
    ]
)

,review_case,content_hash_id,predicted_decline_probability,actual_label,predicted_label,feb_avg_daily_impressions,feb_avg_daily_clicks,feb_avg_position,feb_avg_daily_sessions,feb_engagement_rate,feb_ctr,possible_explanation
0,High-confidence false positive,content_3674ad453e46c6f7,1.000000,0,1,23.464286,0.000000,7.480974,1.000000,0.00,0.000000,"The model assigned high risk because CTR and visibility were comparatively strong, but the page did not decline. Branded queries, stable demand or..."
1,Borderline false negative,content_a60bcdf91647c55a,0.496640,1,0,3.111111,0.037037,8.464286,1.000000,0.00,0.011905,"The model assigned moderate risk just below the decision threshold. A small difference in the score would have changed the class, showing that thi..."
2,High-visibility false negative,content_1073e371db6e53c4,0.306128,1,0,656.178571,0.892857,4.894846,1.388889,0.04,0.001361,"This page declined despite receiving a low predicted probability. Its February CTR pattern may have looked ordinary, while an unobserved factor su..."


### Three concrete wrong cases

#### 1. High-confidence false positive

`content_d95b42029414d8f7` received a predicted decline probability of
0.9991, but it did not decline.

The page had approximately 5.50 average daily impressions, 0.32 average
daily clicks and a February CTR of 5.84%. Because February CTR is the
model's dominant feature, this comparatively high CTR produced a very
strong decline-risk prediction.

However, the page had low traffic volume. A small number of clicks or
impressions can make CTR unstable. Branded searches, stable demand or
temporary query patterns may also explain why the expected decline did
not occur.

#### 2. Borderline false negative

`content_c697dd224ba7d717` received a predicted probability of 0.4985 and
was classified as not declining, but it declined.

This prediction was extremely close to the fixed 0.50 classification
threshold. A very small difference in probability would have changed the
predicted class.

This case shows that class labels can hide uncertainty. The underlying
probability is more informative than treating the result as confidently
negative.

#### 3. High-visibility false negative

`content_c04060d322f34d83` received a predicted probability of 0.4194,
but it declined.

It had approximately 232.18 average daily impressions, making it the most
important missed case among the selected examples. However, its February
CTR was only 0.0923%.

Because the model relies heavily on CTR, this ordinary-looking CTR reduced
the predicted decline risk. The later decline may have been caused by
information unavailable to the model, such as search-demand changes,
seasonality, ranking volatility, query mix or recent content changes.

### Error-analysis conclusion

The preferred model is effective at ranking a small number of high-risk
pages, but it has two important limitations.

First, very high CTR values on low-impression pages can create
overconfident predictions. Second, the model can miss high-visibility
pages when their February CTR does not appear unusual.

The probabilities should therefore support human prioritization rather
than trigger automatic content changes. Business impact, impression
volume and uncertainty should also be considered when reviewing the
ranked recommendations.

In [30]:
import platform
import sklearn

# ---------------------------------------------------------
# Final leakage and reproducibility check
# ---------------------------------------------------------

future_or_target_terms = [
    "march",
    "future",
    "declined",
    "next_month",
    "label",
    "target",
]

leakage_features = [
    feature
    for feature in feature_columns
    if any(
        term in feature.lower()
        for term in future_or_target_terms
    )
]

evaluation_only_columns = [
    "march_avg_daily_clicks",
    "declined_next_month",
]

# Confirm that identifiers are not model inputs.
identifier_features = [
    column
    for column in [
        "client_hash_id",
        "content_hash_id",
        "decision_month",
    ]
    if column in feature_columns
]

# Confirm group separation.
client_overlap = train_clients.intersection(test_clients)

# Confirm row alignment between features, labels and source frames.
train_alignment_ok = (
    X_train.index.equals(y_train.index)
    and X_train.index.equals(train_frame.index)
)

test_alignment_ok = (
    X_test.index.equals(y_test.index)
    and X_test.index.equals(test_frame.index)
)

# Confirm the same test rows were used by all three methods.
same_test_row_count = (
    len(baseline_ranked_test)
    == len(logistic_ranked_test)
    == len(random_forest_ranked_test)
    == len(X_test)
)

same_test_content_ids = (
    set(baseline_ranked_test["content_hash_id"])
    == set(logistic_ranked_test["content_hash_id"])
    == set(random_forest_ranked_test["content_hash_id"])
    == set(test_frame["content_hash_id"])
)

# Read fixed seeds from the fitted workflow.
split_seed = group_splitter.random_state

logistic_seed = (
    logistic_model
    .named_steps["classifier"]
    .random_state
)

forest_seed = random_forest_model.random_state


# ---------------------------------------------------------
# Safety assertions
# ---------------------------------------------------------

assert leakage_features == []
assert identifier_features == []
assert "march_avg_daily_clicks" not in X.columns
assert "declined_next_month" not in X.columns

assert len(client_overlap) == 0
assert train_alignment_ok
assert test_alignment_ok
assert same_test_row_count
assert same_test_content_ids

assert split_seed == RANDOM_STATE
assert logistic_seed == RANDOM_STATE
assert forest_seed == RANDOM_STATE


# ---------------------------------------------------------
# Print the audit result
# ---------------------------------------------------------

print("Final leakage and reproducibility audit")
print("-" * 55)

print("Model features:", feature_columns)
print("Potential leakage features:", leakage_features)
print("Identifier features:", identifier_features)

print(
    "\nEvaluation-only columns:",
    evaluation_only_columns,
)

print(
    "March clicks used as a model feature:",
    "march_avg_daily_clicks" in X.columns,
)

print(
    "Target used as a model feature:",
    "declined_next_month" in X.columns,
)

print("\nOverlapping clients:", len(client_overlap))
print("Training row alignment:", train_alignment_ok)
print("Test row alignment:", test_alignment_ok)

print(
    "Same test-row count for all methods:",
    same_test_row_count,
)

print(
    "Same test content IDs for all methods:",
    same_test_content_ids,
)

print("\nSplit random seed:", split_seed)
print("Logistic Regression seed:", logistic_seed)
print("Random Forest seed:", forest_seed)

print("\nPython version:", platform.python_version())
print("pandas version:", pd.__version__)
print("NumPy version:", np.__version__)
print("scikit-learn version:", sklearn.__version__)

print("\nAll audit checks passed.")
print(
    "The models use February features only, clients do not overlap, "
    "and every method was evaluated on the same held-out rows."
)

Final leakage and reproducibility audit
-------------------------------------------------------
Model features: ['feb_avg_daily_impressions', 'feb_avg_daily_clicks', 'feb_avg_position', 'feb_avg_daily_sessions', 'feb_engagement_rate', 'feb_ctr']
Potential leakage features: []
Identifier features: []

Evaluation-only columns: ['march_avg_daily_clicks', 'declined_next_month']
March clicks used as a model feature: False
Target used as a model feature: False

Overlapping clients: 0
Training row alignment: True
Test row alignment: True
Same test-row count for all methods: True
Same test content IDs for all methods: True

Split random seed: 42
Logistic Regression seed: 42
Random Forest seed: 42

Python version: 3.12.13
pandas version: 2.2.2
NumPy version: 2.0.2
scikit-learn version: 1.6.1

All audit checks passed.
The models use February features only, clients do not overlap, and every method was evaluated on the same held-out rows.


### Leakage and reproducibility conclusion

The final audit found no future-data or target leakage.

The models use six features created only from February information:

- Average daily impressions
- Average daily clicks
- Average search position
- Average daily sessions
- Engagement rate
- Click-through rate

March average daily clicks and `declined_next_month` were used only for
evaluation. They were not included in the model feature matrix.

Client and content identifiers were also excluded from the predictive
features. `client_hash_id` was used only to create the grouped split.

No client appeared in both training and testing. The Week 4 baseline,
Logistic Regression and Random Forest were evaluated using the same
1,672 held-out test rows.

A fixed random seed of 42 was used for the grouped split, Logistic
Regression and Random Forest. The Python and library versions were also
recorded to support reproducibility.

All leakage, alignment, group-separation and same-test-row checks passed.

The final results should still be interpreted as observed performance on
five held-out clients rather than guaranteed performance for every future
client.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.